# B2-024 — Session 3: Evaluating Open-Ended Models

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

A single number on a single run is a measurement with unknown error bars.
This session gives you the tools to attach honest uncertainty to a score and to compare models fairly: choosing the metric, the variance of a mean, the percentile bootstrap, paired comparisons, ablation tables, robustness checks under noise and shift, comparisons across model families, and the written evaluation section.
The probability you need is Book 1's `book1:expectation`, `book1:variance`, `book1:variance-of-sums`, `book1:independence`, and `book1:covariance`; the bootstrap itself is new and is taught here.

## 1. Choosing the metric

The metric is part of the task definition, so read it before you design anything.
This unit uses three, each an average of a **per-example** quantity:

* **Accuracy** for shape classification: the mean of $\mathbb 1[\hat y_i=y_i]$.
* **Mean source-position error** for the inverse problem: the mean of $\lVert(\hat x_i,\hat y_i)-(x_i,y_i)\rVert_2$; the predicted strength is not scored, but a physics residual (Session 5) checks it.
* **Mean permutation-invariant parameter error** for mixtures: for each example, the minimum over the six relabelings of the three components of the mean absolute difference over the nine parameters (Session 6).

`capstone_data.metric_per_example(task, pred, target)` returns these per-example values; the dataset score is their mean.
Keeping the per-example values (not only the mean) is what makes bootstrap intervals and paired comparisons possible.

**When accuracy misleads.** With imbalanced classes (`book1:class-imbalance`), accuracy rewards predicting the majority class; per-class **precision** and **recall** (`book1:accuracy-precision-recall`) expose it.
A pseudo-labeling run that floods one class (Session 4) can raise accuracy on a majority-heavy split while destroying recall on the others.
Report the per-class confusion when classes matter.

**Mean versus tail.** A mean error hides a few catastrophic failures; the mixture least-squares baseline, for example, fits most functions well and a few very badly (a bump stuck far from its true center).
When failures matter, also report a high quantile (the 90th percentile error) or the fraction of examples above a tolerance.
Whatever you report, the **one** score you optimize on validation must be the task's metric; secondary numbers are for understanding, not selection.

**Checkpoint 1A.** Three test examples have position errors $0.01$, $0.02$, $0.30$. Give the mean error and say what it hides.

**Checkpoint 1B.** A classifier on a split with 80% class 0 always predicts class 0. What are its accuracy and its recall on class 1?

**Collected answers.** The mean is $0.11$; it hides that two examples are excellent and one is a gross failure (a median or tail quantile would show it). Accuracy $0.8$; recall on class 1 is $0$.

## 2. The variance of a mean, and why one run is not enough

Let $X_1,\dots,X_n$ be per-example values with common mean $\mu$ and variance $\sigma^2$, and let $\bar X=\frac1n\sum_i X_i$.
By linearity of expectation (`book1:expectation`), $E[\bar X]=\mu$.
For the variance, `book1:variance-of-sums` gives
$$\operatorname{Var}\Big(\sum_i X_i\Big)=\sum_i\operatorname{Var}(X_i)+\sum_{i\ne j}\operatorname{Cov}(X_i,X_j).$$
If the $X_i$ are **independent** (`book1:independence`), every covariance is $0$, so the sum has variance $n\sigma^2$, and since $\operatorname{Var}(aY)=a^2\operatorname{Var}(Y)$,
$$\operatorname{Var}(\bar X)=\frac{1}{n^2}\cdot n\sigma^2=\frac{\sigma^2}{n},\qquad \operatorname{sd}(\bar X)=\frac{\sigma}{\sqrt n}.$$
This **standard error** shrinks like $1/\sqrt n$: four times as many test examples halve the uncertainty.

**Two sources of randomness.** A reported score varies because (a) the *test examples* are a sample from a larger population, and (b) the *training run* depends on seeds.
The formula above describes (a), with $n$ the number of examples; the same formula applies to (b) with $n$ the number of seeds and $X_i$ the per-seed scores.
They are different quantities: a tight interval over test examples says nothing about how a different seed would have scored, and vice versa.
Practice p15 asks you to prove the formula and its paired-difference version.

**Example.** 300 test images with accuracy around $0.85$: each $X_i\in\{0,1\}$ has variance $p(1-p)\approx0.1275$, so the standard error is $\sqrt{0.1275/300}\approx0.021$.
A difference of $0.01$ between two models on this test set is half a standard error, well within noise.

**Checkpoint 2A.** Per-example errors have standard deviation $0.012$ over $n=200$ examples. What is the standard error of the mean error?

**Checkpoint 2B.** Why does $\operatorname{Var}(\bar X)=\sigma^2/n$ fail if the examples are near-duplicates of each other?

**Collected answers.** $0.012/\sqrt{200}\approx0.00085$. Near-duplicates are positively correlated, so the covariance terms are positive and the variance of the mean is larger than $\sigma^2/n$; the effective sample size is smaller than $n$.

## 3. The percentile bootstrap

The formula $\sigma/\sqrt n$ needs an estimate of $\sigma$ and an approximately normal mean.
The **bootstrap** avoids both by simulating the sampling process with the data you have.

**Algorithm (percentile bootstrap for a metric over $n$ examples).**

1. Compute the point estimate $\hat\theta=\text{metric}(y,\hat y)$ on all $n$ examples.
2. For $b=1,\dots,B$: draw $n$ indices **with replacement** from $\{0,\dots,n-1\}$, and compute $\theta^*_b=\text{metric}(y[\text{idx}],\hat y[\text{idx}])$ on the resampled examples, **resampling labels and predictions together**.
3. The $(1-\alpha)$ percentile interval is the $\alpha/2$ and $1-\alpha/2$ quantiles of $\theta^*_1,\dots,\theta^*_B$.

In this unit the procedure is pinned so that results are reproducible:
`idx = torch.randint(0, n, (n_boot, n), generator=generator)` draws all indices in one call from a **seeded** `torch.Generator`, and the quantiles use `torch.quantile` (linear interpolation) on a `float64` tensor of the $\theta^*_b$.
`capstone_data.final_test_score` uses exactly this, with `n_boot=1000`, `alpha=0.05`, and a generator seeded from the unit seed; Practice p08 asks you to implement it as `bootstrap_ci`.

**What the interval means.** It approximates how much the score would move if the test examples were re-drawn from the same population; it does not include seed variability, and it is not a guarantee about any single future example.

**A classic bug.** Resampling the predictions but keeping the labels in their original order pairs each prediction with the wrong label, which destroys the per-example structure; the "interval" then measures noise around chance agreement rather than the model's score.
Always index both with the same `idx`.

**Checkpoint 3A.** Why must the bootstrap draw indices with replacement rather than permute the examples?

**Checkpoint 3B.** With `n_boot=1000` and `alpha=0.1`, which quantiles of the resampled statistics form the interval?

**Collected answers.** A permutation of all $n$ examples gives exactly the same mean every time (zero spread); sampling with replacement varies how often each example appears, which mimics drawing a new sample from the population. The $0.05$ and $0.95$ quantiles.

## 4. Paired comparisons

To compare models $A$ and $B$ on the **same** test examples, look at the per-example differences $D_i=X^A_i-X^B_i$ rather than at two separate intervals.
By `book1:variance-of-sums` with a covariance term (`book1:covariance`),
$$\operatorname{Var}(X^A_i-X^B_i)=\operatorname{Var}(X^A_i)+\operatorname{Var}(X^B_i)-2\operatorname{Cov}(X^A_i,X^B_i).$$
Two models evaluated on the same examples are usually **positively correlated** (both find the same examples hard), so the covariance term makes the difference much less variable than either score.
Two overlapping separate intervals can therefore hide a difference that a paired interval shows clearly.

**Paired bootstrap.** Use one shared `idx` for both models in every bootstrap replicate and record $\theta^*_b(A)-\theta^*_b(B)$; the percentile interval of these differences is the paired interval.
If it excludes $0$, the difference is unlikely to be an artifact of which examples happened to be in the test set.

**Pairing across seeds.** The same idea applies to training randomness: train $A$ and $B$ with the same seed list, compute per-seed differences $d_s=s^A_s-s^B_s$, and report their mean and sample standard deviation.
A difference whose per-seed values all have the same sign is far more convincing than two overlapping mean ± std bars.

**Multiple comparisons.** Each comparison you make has some chance of a spurious "win"; if you compare ten variants against a baseline at $\alpha=0.05$, about one may look significant by luck.
Pre-register the comparisons that matter, and treat surprising wins from a large search as hypotheses for the next experiment, not conclusions.

**Checkpoint 4A.** $\operatorname{Var}(X^A)=\operatorname{Var}(X^B)=0.04$ and $\operatorname{Cov}(X^A,X^B)=0.03$. Compare $\operatorname{Var}(X^A-X^B)$ with the independent case.

**Checkpoint 4B.** Model $A$'s 95% interval is $[0.80, 0.88]$ and $B$'s is $[0.83, 0.91]$. Can you conclude anything about $B-A$ from these alone?

**Collected answers.** Paired: $0.04+0.04-0.06=0.02$; independent: $0.08$, four times larger. No: overlapping separate intervals neither prove nor rule out a difference, because they ignore the correlation; compute the paired interval of the per-example differences.

## 5. Ablation tables and seed statistics

An **ablation table** has one row per configuration and columns for the statistics a reader needs to judge differences:

| config | change from reference | seeds | steps/run | mean val error | sample std | per-seed diff vs reference |
|---|---|---:|---:|---:|---:|---|
| A | reference | 3 | 250 | 0.0234 | 0.0019 | — |
| B | without log features | 3 | 250 | 0.0291 | 0.0024 | +0.0049, +0.0061, +0.0061 |
| C | width 32 instead of 64 | 3 | 250 | 0.0241 | 0.0021 | +0.0012, −0.0004, +0.0013 |

Reading it: removing log features hurts by about $0.0057$, about $2.5$ standard deviations of the per-config spread, with all three per-seed differences positive, so log features earn their place.
Width 32 differs from width 64 by $0.0007$ with mixed-sign per-seed differences: not demonstrated.

**Rules for an honest table.** The same seeds and steps in every row; validation numbers only (the test set is not a table column); the sample standard deviation (divisor $r-1$); the number of seeds stated; and per-seed differences when the comparison matters.
A reader should be able to recompute every mean from the per-seed numbers you log.

**What a table cannot show.** A table over three seeds and one validation set measures seed noise and nothing about distribution shift; Section 6 adds that axis.

**Checkpoint 5A.** In the table, why is row C's difference "not demonstrated" even though its mean is worse?

**Checkpoint 5B.** A table reports `std` computed with `torch.std(scores, unbiased=False)` over two seeds. By what factor is it too small?

**Collected answers.** Its mean difference ($0.0007$) is small compared with the seed spread and the per-seed differences change sign, so seed noise alone could produce it. For $r=2$ the population formula divides by $2$ instead of $1$, so the reported standard deviation is smaller by a factor $\sqrt{2}\approx1.41$.

## 6. Robustness, model families, and writing the evaluation section

**Robustness to noise and shift.** A validation set drawn from the training distribution measures in-distribution performance.
Deployment conditions differ: sensors get noisier, images get dimmer, sources drift toward the domain edge.
A **robustness check** re-evaluates the trained model on validation inputs perturbed in a stated, seeded way, for example multiplying every sensor reading by $1+0.05\varepsilon$ with $\varepsilon\sim N(0,1)$ from a fixed generator, and reports the score change.
Two models with equal clean scores can differ sharply under shift; a model that leans on one sensor degrades when that sensor is noisy.
Shift checks are evaluations, never training data unless the method says so.

**Comparing model families.** Comparing, say, a shallow-wide MLP with a deep-narrow one is fair only when each gets the same data, budget, seeds, and tuning effort.
Report parameter counts (`book1:parameter-counting`) and wall-clock time beside the scores; a family that wins by $1\%$ at ten times the cost may not be the right choice for a timed round.
Choose the family with a **pre-registered rule** applied to validation numbers (for example: lowest mean validation error over shared seeds; if the paired difference's interval contains $0$, choose the cheaper family).

**The evaluation section** (`book1:writeup-quality`) of an open-ended writeup answers five questions in order:

1. *Protocol*: which rows trained, which selected, how the test set was locked, which seeds and budget.
2. *Headline*: the single final test score with its bootstrap interval and $n$, and the baseline's score under the same protocol.
3. *Evidence for choices*: the validation ablation table and the paired comparison that justified the final configuration.
4. *Robustness*: the shift checks and what they reveal.
5. *Limitations*: what the numbers do not show (seed variability not in the test interval, one test set, a weak baseline).

Every number in the section should be traceable to the log.

**Checkpoint 6A.** Why must a robustness perturbation use a seeded generator?

**Checkpoint 6B.** Which number in the evaluation section comes from the locked test set, and how many times was it computed?

**Collected answers.** So that both models see exactly the same perturbed inputs (a paired comparison) and the check is reproducible; unseeded noise adds its own variability to the comparison. Only the headline final score (with its bootstrap interval), computed once.

## 7. Worked laboratory: two families, intervals, a paired difference, and one test call

The laboratory compares two small MLP families on the inverse task, under one protocol: training rows only, 200 full-batch Adam steps, shared seeds 0 and 1, log-magnitude features computed inside `forward`.
Family `W16` has one hidden layer of width 16; family `W64` has one hidden layer of width 64.
All comparisons use validation rows: a bootstrap interval for each family (seed 0), a paired bootstrap interval of the difference, per-seed differences, and a robustness check under 5% multiplicative sensor noise.
The pre-registered rule chooses the family with the lower mean validation error, preferring the smaller family when the paired interval contains $0$.
Only then is the chosen model scored once with `final_test_score`.

**Checkpoint 7A.** Which object guarantees that both families' bootstrap replicates use the same examples in the paired interval?

**Checkpoint 7B.** Why is the test score printed only after the selection rule has run?

**Collected answers.** One shared index tensor `idx` (from one seeded generator) used to index both families' per-example errors in every replicate. Because a test score seen before selection could influence the choice, turning the test set into a selection set; scoring after selection keeps it an unbiased one-shot estimate.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

train_x, train_y = capstone_data.train_rows("inverse")
val_x, val_y = capstone_data.val_rows("inverse")
log_train = torch.log(train_x)

class LogMLP(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.register_buffer("mu", log_train.mean(dim=0))
        self.register_buffer("sd", log_train.std(dim=0))
        self.net = nn.Sequential(nn.Linear(8, hidden), nn.ReLU(), nn.Linear(hidden, 3))

    def forward(self, magnitudes):
        return self.net((torch.log(magnitudes) - self.mu) / self.sd)

def fit(hidden, seed, steps=200):
    torch.manual_seed(seed)
    model = LogMLP(hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)
        F.mse_loss(model(train_x), train_y).backward()
        optimizer.step()
    return model.eval()

def val_errors(model, inputs=val_x):
    with torch.no_grad():
        return capstone_data.position_error_per_example(model(inputs), val_y)

families = {"W16": 16, "W64": 64}
seeds = (0, 1)
models = {(name, s): fit(h, s) for name, h in families.items() for s in seeds}
per_seed = {name: [val_errors(models[(name, s)]).mean().item() for s in seeds] for name in families}
print({k: [round(v, 5) for v in vals] for k, vals in per_seed.items()})

In [ ]:
def percentile_interval(stats, alpha):
    q = torch.tensor([alpha / 2, 1 - alpha / 2], dtype=torch.float64)
    return [round(v, 5) for v in torch.quantile(stats.double(), q).tolist()]

n_boot, alpha = 1000, 0.05
err16, err64 = val_errors(models[("W16", 0)]), val_errors(models[("W64", 0)])
idx = torch.randint(0, err16.shape[0], (n_boot, err16.shape[0]), generator=torch.Generator().manual_seed(SEED))
print("W16 seed-0 val error", round(err16.mean().item(), 5), "CI", percentile_interval(err16[idx].mean(dim=1), alpha))
print("W64 seed-0 val error", round(err64.mean().item(), 5), "CI", percentile_interval(err64[idx].mean(dim=1), alpha))
paired = (err64[idx] - err16[idx]).mean(dim=1)                 # same idx for both: paired
print("paired W64 - W16:", round((err64 - err16).mean().item(), 5), "CI", percentile_interval(paired, alpha))
print("per-seed differences W64 - W16:", [round(a - b, 5) for a, b in zip(per_seed["W64"], per_seed["W16"])])

shift_gen = torch.Generator().manual_seed(SEED + 7)
shifted = val_x * (1 + 0.05 * torch.randn(val_x.shape, generator=shift_gen))
for name in families:
    clean = val_errors(models[(name, 0)]).mean().item()
    noisy = val_errors(models[(name, 0)], shifted).mean().item()
    print(f"{name}: clean {clean:.5f} -> 5% sensor noise {noisy:.5f}")

In [ ]:
mean_err = {name: sum(v) / len(v) for name, v in per_seed.items()}
paired_ci = percentile_interval(paired, alpha)
if paired_ci[0] <= 0.0 <= paired_ci[1]:
    chosen = "W16"                                      # interval contains 0: prefer the smaller family
else:
    chosen = min(mean_err, key=mean_err.get)
print("chosen family:", chosen, "| parameter counts:",
      {name: sum(p.numel() for p in models[(name, 0)].parameters()) for name in families})

final_model = models[(chosen, 0)]
def predict(magnitudes):
    with torch.no_grad():
        return final_model(magnitudes)
result = capstone_data.final_test_score(predict, n_boot=1000, alpha=0.05, task="inverse")
print(result)
print("baseline:", round(capstone_data.baseline_score("inverse").score, 4), "| test calls:", capstone_data.test_call_count("inverse"))

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Optimizing a metric other than the task's.* Fix: select on the task metric; report others for understanding only.
2. *A bootstrap that resamples predictions but not labels.* Fix: one `idx` for both.
3. *An unseeded bootstrap.* Fix: a dedicated `torch.Generator` with a stated seed; pinned quantile method.
4. *Comparing overlapping separate intervals.* Fix: a paired interval of differences on shared examples and seeds.
5. *Reading test-example intervals as seed intervals.* Fix: report both kinds of variability separately.
6. *Population standard deviation over seeds.* Fix: divisor $r-1$.
7. *Shift checks with fresh random noise for each model.* Fix: one seeded perturbation shared by all models.
8. *An evaluation section without the protocol or the baseline under the same protocol.* Fix: the five-part structure.

**Exam connections.** p08 implements `bootstrap_ci` and `paired_bootstrap_diff`; p15 proves $\operatorname{Var}(\bar X)=\sigma^2/n$ and the paired-difference variance reduction; p21 writes a full evaluation report for two model families with bootstrap intervals, a paired comparison across shared seeds, an ablation table, and a noise-shift check; p24 decides which differences in a supplied results table are real; p28 audits a report for test-set reuse, seed cherry-picking, an invalid interval, and leakage.

**Checkpoint 8A.** A report's 95% "bootstrap" interval over 200 test examples is narrower than the spread of its five seed scores. Is that a contradiction?

**Checkpoint 8B.** Which practice asks you to audit an invalid confidence interval?

**Collected answers.** No: the bootstrap interval measures test-example sampling for one trained model, while the seed spread measures training randomness; they are different sources and need not agree. p28.

## 9. Forward-only deeper topic: beyond the percentile bootstrap

Forward-only deeper topic.
The percentile interval can be biased when the statistic's distribution is skewed; the **BCa** (bias-corrected and accelerated) bootstrap adjusts the quantiles for bias and skewness.
For comparing two classifiers on the same examples, **McNemar's test** uses only the examples where they disagree.
**Calibration** asks whether a model's predicted probabilities match observed frequencies (a reliability diagram plots one against the other), which matters for the confidence thresholds of Session 4.
**Conformal prediction** turns any model into prediction sets with a guaranteed coverage rate under exchangeability, using a held-out calibration split.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-024.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Which examples does McNemar's test use?

**Checkpoint 9B.** Why would calibration matter for confidence-thresholded pseudo-labels?

**Collected answers.** Only those where the two classifiers disagree (one right, one wrong). A threshold such as $0.9$ on predicted probability assumes the probabilities mean something; an overconfident model passes many wrong pseudo-labels through the threshold.